In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sentiment-analysis-dataset/Dropbox.csv


In [2]:
!pip install transformers


In [3]:
import torch

In [4]:
df=pd.read_csv("/kaggle/input/sentiment-analysis-dataset/Dropbox.csv")
df.tail(20)

,reviewId,content,score
9980,e749c2aa-a006-4dcc-be92-c21d1d5b064a,Love this app had it for years.,5
9981,6ea8f9d4-4e7d-460d-b42a-89c11d5ec4d4,Ni e,5
9982,be862866-076e-48a9-80e6-58311ce59634,😒,1
9983,1f47539e-1657-4fa2-9f5b-e1d684675a6d,WHERE'S MY MUSIC? IT WAS THERE BEFORE?,3
9984,3db5cddd-da78-4c58-abb6-90631de46f25,very good application,5
9985,4b7e26f5-feb0-4868-8b31-9caf97ca7a47,Where is all my.files ...this this was so full...,2
9986,50145bb0-aa94-4913-9171-64a82699999f,Simple great,5
9987,0c9cf5a6-9704-45b0-8d20-6d5fa796c921,Really frustrating! I am being charged TWICE. ...,3
9988,b6c7db05-943c-471b-9fc7-30bdacda8c38,Wonderful,5
9989,be2e8bc2-0e9b-4849-9f55-bafc8904eb09,Easy to use. No problems so far :),5


In [5]:
df.describe

<bound method NDFrame.describe of                                   reviewId  \
0     dafa8ee2-babc-423b-870e-2d9279cff01e   
1     b4f40c25-41b5-4c97-b4cb-d35930a609e0   
2     5505adff-9bae-439e-b242-698ea2a8482c   
3     e6b4e364-30a8-423d-bf16-8aedd8493235   
4     70257417-02ed-47a1-b1bb-b0079e9c86ae   
...                                    ...   
9995  1723b8cb-b5b7-49db-94a0-b8185f2007e6   
9996  4f3288ed-6800-4f22-91f9-eb321a958487   
9997  9a694712-d3b9-46b7-bcec-c3c0a7207029   
9998  c96bd9ea-39b8-49f7-bb5d-de8f38a1adfc   
9999  dfd445a2-0681-44dc-b446-e0abc0e70d59   

                                                content  score  
0     It's exactly what I need. A place for me to or...      5  
1     تضمن كل ملفاتك وصورك محفوظه من التلف او الضياع...      5  
2                                Meta WhatsApp business      5  
3     Kudos to Dropbox app for an effective cloud st...      5  
4            Great options for backups! plenty of space      5  
...                    

In [6]:
df=df.sample(200)
df

,reviewId,content,score
2792,7c252e16-0b77-4f93-9e4e-a01ce6bc0202,Good,5
6845,621fa3e4-35cb-4bd1-8265-f685a532cf77,I downloaded it as a photo backup service but ...,2
5334,6d1a8856-6dfd-45c8-87e5-6728f3fd10fc,Good,5
5202,cf318102-e2dd-4ba9-8d26-6902d3a366ca,😁😁😁,3
4593,9594eed6-2bf5-4f78-b490-28461f03faac,The Latest big Changes made a perfectly good a...,1
...,...,...,...
1000,41e86cdc-93eb-46ac-846c-41801d96128c,About photo,1
5972,e7b8d21e-4552-49d9-ae69-154108727e34,showing space used 13 GB vs actual use of 1.3 gb,3
3745,e27d2a47-2f3e-430d-847a-c91a8c650b8d,I signed up for a free trial with an upgrade t...,2
8484,f09c3440-ed1f-4dd3-b272-4c0e7b67950a,Good,5


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline



In [11]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to("cuda")
nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)


In [12]:
label_mapping = {
    "LABEL_0": "negative",
    "LABEL_1": "neutral",
    "LABEL_2": "positive"
}
texts = list(df.content.values)
results = nlp(texts)

for text, result, score in zip(texts, results, df.score.values):
    human_label = label_mapping[result["label"]]  
    print("Text:", text)
    print("Result:", {"label": human_label, "score": result["score"]})
    print("Score:", score)
    print("-" * 80)

Text: Good
Result: {'label': 'positive', 'score': 0.6097784638404846}
Score: 5
--------------------------------------------------------------------------------
Text: I downloaded it as a photo backup service but it doesn't seem to work right. It will backup photos for a while and then just stop. Since that was my primary reason for using the app, and it doesn't work, I give it a low rating and don't trust it with my precious photos.
Result: {'label': 'negative', 'score': 0.9294798970222473}
Score: 2
--------------------------------------------------------------------------------
Text: Good
Result: {'label': 'positive', 'score': 0.6097784638404846}
Score: 5
--------------------------------------------------------------------------------
Text: 😁😁😁
Result: {'label': 'positive', 'score': 0.8840939402580261}
Score: 3
--------------------------------------------------------------------------------
Text: The Latest big Changes made a perfectly good app almost useless. Before I could open any 